## Как писать Scorer?

[Общие требования к коду](https://confluence.delta.sbrf.ru/pages/viewpage.action?pageId=12039618894)

Все scorer'ы должны находиться в папке `llm_val/scorer` и наследоваться от родительского класса `Scorer`, который лежит в файле `llm_val/scorer/Scorer.py`. У scorer'а есть атрибут `metrics` - словарь, где ключ - это название метрики, а значение - словарь с двумя обязательными ключами: `call`(`Callable`) и `is_singlecol`(`bool`); и метод `calc`, который считает метрики.

Структура атрибута `metrics`:
```python
{'single_mean': {'call': lambda x: np.mean(x.values), 'is_singlecol': True}}
```
Сигнатура метода `calc`:
```python
def calc(self,
         sampler: Sampler,
         data_type: str,
         metric_name: str) -> tp.Dict[str, float]:
```
Метод `calc` должен вернуть словарь с посчитанными метриками, где ключ - название метрики, а значение - значение метрики.

В отличие от sampler'а, scorer является довольно простой в имплементации абстракцией. Все, что надо сделать пользователю, чтобы написать scorer - это переопределить метод `calc`.

### Этапы написания scorer'a

1. **Создаем класс**. Называем его как-нибудь осмыслленно и переопределяем конструктор.
2. **Пишем логику подсчета метрик**. На данном этапе пользователь может написать любую реализацию: главное, чтобы конечный вывод совпадал с сигнатурой метода.

Опциональные пункты, если вы хотите добавить свой scorer в качестве примера в `llm_val`:

3. **Пишем докстринги**. Комментарии являются важной частью при написании любого кода. [Требования к комментариям (Раздел "Документация")](https://confluence.delta.sbrf.ru/pages/viewpage.action?pageId=12039618894).
4. **Пишем юнит-тесты**. Иерархия юнит-тестов повторяет иерархию библиотеки. Тесты по scorer'ам находятся в файле `unit_tests/scorer/test_scorer.py`.

## Пример с рассуждениями

Попробуем написать свой scorer, используя план выше. Допустим, что необходимо написать универсальный scorer для MMLU и RuBQ датасетов.

### 1. Создаем класс

Так как у нас универсальный scorer для нескольких типов датасетов, то назовем его `UniversalScorer`. В конструкторе просто передадим словарь с метриками в родительский конструктор:

```python
class UniversalScorer(Scorer):
    def __init__(self,
                 metrics: tp.Dict[str, tp.Any]):
        super().__init__(metrics)
```

### 2. Пишем логику подсчета метрик

Для примера напишем метод, который будет считать конкретную метрику и возвращать словарь с одним ключ-значением. Для этого в параметр `metric_name` будем передавать название метрики, котороую хотим посчитать. Если значение `None`, то считаем все доступные метрики.

Получаем следующий код:
```python
def calc(self,
         sampler: Sampler,
         data_type: str='test',
         metric_name: tp.Optional[str]=None) -> tp.Dict[str, float]:
    metrics = self.metrics
    data = getattr(sampler, data_type)['y']
    if metric_name is not None:
        metrics = {metric_name: self.metrics[metric_name]}
    metric_res = {}
    for func_name, func_dict in metrics.items():
        if func_dict['is_singlecol']:
            for name_col in data:
                name = f'{func_name}_{name_col}'
                metric_res[name] = func_dict['call'](data[name_col])
        else:
            metric_res[func_name] = func_dict['call'](data)
    return metric_res
```

Теперь код scorer'а выглдяит так:
```python
class UniversalScorer(Scorer):
    def __init__(self,
                 metrics: tp.Dict[str, tp.Callable]):
        super().__init__(metrics)


    def calc(self,
             sampler: Sampler,
             data_type: str='test',
             metric_name: tp.Optional[str]=None) -> tp.Dict[str, float]:
        metrics = self.metrics
        data = getattr(sampler, data_type)['y']
        if metric_name is not None:
            metrics = {metric_name: self.metrics[metric_name]}
        metric_res = {}
        for func_name, func_dict in metrics.items():
            if func_dict['is_singlecol']:
                for name_col in data:
                    name = f'{func_name}_{name_col}'
                    metric_res[name] = func_dict['call'](data[name_col])
            else:
                metric_res[func_name] = func_dict['call'](data)
        return metric_res
```

### 3. Пишем докстринги

Самая важная часть написания скорера для библиотеки - это написание документации. Подробнее посмотреть требования к документации можно [здесь (Раздел "Документация")](https://confluence.delta.sbrf.ru/pages/viewpage.action?pageId=12039618894).
Код с комментариями выглядит следующим образом:
```python
class UniversalScorer(Scorer):
    def __init__(self,
                 metrics: tp.Dict[str, tp.Any]):
        '''
        metrics: словарь, где ключ - название метрики, 
                              значение - словарь с двумя обязательными ключами: `call`(`Callable`) и `is_singlecol`(`bool`)
        '''
        super().__init__(metrics)


    def calc(self,
             sampler: Sampler,
             data_type: str='test',
             metric_name: tp.Optional[str]=None) -> tp.Dict[str, float]:
        '''
        Метод для подсчета метрик
        Parameters:
            sampler: Объект класса Sampler
            data_type: Тип данных, на которых производится расчет метрики. Hint: название атрибута в sampler'е
            metric_name: Название возвращаемой метрики
        Return:
            Словарь с посчитанными метриками
        '''
        metrics = self.metrics
        data = getattr(sampler, data_type)['y']
        if metric_name is not None:
            metrics = {metric_name: self.metrics[metric_name]}
        metric_res = {}
        for func_name, func_dict in metrics.items():
            if func_dict['is_singlecol']:
                for name_col in data:
                    name = f'{func_name}_{name_col}'
                    metric_res[name] = func_dict['call'](data[name_col])
            else:
                metric_res[func_name] = func_dict['call'](data)
        return metric_res
        
```

### 4. Пишем юнит-тесты

Вторая самая важная часть в написании scorer'a для библиотеки - это написание юнит-тестов. Все файлы с юнит-тестами получают префикс test_.

Обязательные(минимальные) проверки в юнит-тестах:
1. Проверка на корректность создания scorer'а;
2. Проверка на корректность посчитанных метрик.

Пример юнит-тестов можно посмотреть в `unit_tests/scorer/test_scorer.py`